# ADV-06: Robust Visual Perception in Adverse Conditions
## Atmospheric Scattering, Dark Channel Prior, and Neural Uncertainty Calibration

**Author:** Aji Pamungkas Tri Nurcahyo (Institut Teknologi Sumatera - ITERA)  
**Level:** L3 Advanced Curriculum  
**Pillar:** Pillar 2 — Intelligent Visual Perception & Sensing (ISVWS 2026–2040)  

---
### Abstract
This notebook explores visual perception degradation in harsh environmental conditions:
1. Simulation of Koschmieder's physical atmospheric radiative transfer model $I(x) = J(x) t(x) + A (1 - t(x))$.
2. Dark Channel Prior (DCP) transmission map estimation and single-image dehazing restoration.
3. Objective image quality assessment: PSNR, SSIM, and MSE.
4. Safety-critical neural uncertainty calibration via Expected Calibration Error (ECE).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print(f"NumPy Version: {np.__version__}")

### 1. Koschmieder Atmospheric Scattering Simulation
$$I(x) = J(x) \exp(-\beta d(x)) + A (1 - \exp(-\beta d(x)))$$

In [ ]:
h, w = 64, 64
clean = np.zeros((h, w, 3), dtype=np.float64)
clean[:h//2, :, 0] = 60; clean[:h//2, :, 1] = 140; clean[:h//2, :, 2] = 220 # Sky
clean[h//2:, :, 0] = 50; clean[h//2:, :, 1] = 170; clean[h//2:, :, 2] = 80  # Ground

y_depth = np.linspace(0.1, 1.0, h)[:, np.newaxis]
depth_map = np.tile(y_depth, (1, w))
beta = 0.8
airlight = 230.0
transmission = np.exp(-beta * 2.5 * depth_map)

hazy = clean * transmission[:, :, np.newaxis] + airlight * (1.0 - transmission[:, :, np.newaxis])
hazy = np.clip(hazy, 0.0, 255.0)

mse_hazy = np.mean((clean - hazy)**2)
psnr_hazy = 10 * np.log10((255.0**2) / mse_hazy)
print(f"Hazy Scene PSNR: {psnr_hazy:.2f} dB (Severe Contrast Loss)")

### 2. Dark Channel Prior Transmission Map & Radiance Recovery
$$J^{\text{dark}}(x) = \min_{c} \left( \min_{y \in \Omega(x)} I^c(y) \right)$$
$$J(x) = \frac{I(x) - A}{\max(t(x), t_0)} + A$$

In [ ]:
min_c = np.min(hazy, axis=2)
pad = 2
padded = np.pad(min_c, pad, mode='edge')
dark_chan = np.zeros((h, w))
for y in range(h):
    for x in range(w):
        dark_chan[y, x] = np.min(padded[y:y+5, x:x+5])

t_est = np.maximum(0.15, 1.0 - 0.95 * (dark_chan / airlight))
restored = (hazy - airlight) / t_est[:, :, np.newaxis] + airlight
restored = np.clip(restored, 0.0, 255.0)

mse_restored = np.mean((clean - restored)**2)
psnr_restored = 10 * np.log10((255.0**2) / mse_restored)
print(f"Restored Scene PSNR: {psnr_restored:.2f} dB")
print(f"PSNR Gain: +{psnr_restored - psnr_hazy:.2f} dB")
assert psnr_restored > psnr_hazy, "Restoration must improve PSNR!"

### 3. Neural Calibration Evaluation (Expected Calibration Error - ECE)
$$\text{ECE} = \sum_{m=1}^M \frac{|B_m|}{N} |\text{acc}(B_m) - \text{conf}(B_m)|$$

In [ ]:
np.random.seed(42)
n = 150
# Clean In-Distribution
c_conf = np.random.uniform(0.6, 0.95, n)
c_acc = (np.random.rand(n) < c_conf).astype(np.float64)

# Adverse Out-of-Distribution (Miscalibrated Overconfidence)
a_conf = np.random.uniform(0.75, 0.95, n)
a_acc = (np.random.rand(n) < 0.50).astype(np.float64)

def calc_ece(conf, acc, bins=5):
    edges = np.linspace(0.0, 1.0, bins + 1)
    err = 0.0
    for i in range(bins):
        m = (conf >= edges[i]) & (conf <= edges[i+1])
        if np.sum(m) > 0:
            err += (np.sum(m) / len(conf)) * np.abs(np.mean(acc[m]) - np.mean(conf[m]))
    return err

ece_in = calc_ece(c_conf, c_acc)
ece_out = calc_ece(a_conf, a_acc)
print(f"In-Distribution Calibration Error: {ece_in * 100:.2f}%")
print(f"Adverse OOD Miscalibration Error:   {ece_out * 100:.2f}%")
assert ece_out > ece_in, "Adverse conditions exhibit higher miscalibration!"
print("[PASSED] Uncertainty calibration verified!")

### 4. Summary & Research Connections
- Under adverse weather, restoration alone does not solve the domain shift problem.
- Direct connection to **Track R03 (Robust Visual Perception in Adverse Environments)** and safety-critical calibration.